# ETL Bronze to Silver - VERSÃO OTIMIZADA PARA BI

**Objetivo de Negócio:** Identificar produtos com melhor desempenho de vendas para decisões de estoque, promoções e destaque  
**Cliente:** Gerentes de E-commerce e Vendedores Amazon

## Principais Features para Business Intelligence:
- ✅ **Deduplicação baseada em ASIN** (código único do produto)
- ✅ **Extração automática de marca** a partir dos títulos
- ✅ **Inferência de categoria** do produto
- ✅ **Preenchimento inteligente de preços** faltantes (salva 27.5% dos dados)
- ✅ **Faixas de preço** (Budget, Economy, Premium, Luxury)
- ✅ **Quality Score** (métrica composta: rating + reviews)
- ✅ **Receita estimada** por produto
- ✅ **Flag de produto promovível** (critérios de qualidade)
- ✅ **Engenharia de descontos e cupons**
- ✅ **Features temporais** (hora, dia da semana, etc.)

## 🎯 Diferencial desta versão:
- **Orientado para BI/Power BI** (não para Machine Learning)
- **Métricas de negócio** ao invés de transformações estatísticas
- **Nomes de colunas claros** para facilitar dashboards
- **Flags acionáveis** (ex: is_promotable)

---

### 📖 Como usar este notebook:

1. **Execute as células em ordem** - cada etapa depende da anterior
2. **Dados de entrada:** arquivo CSV bruto na pasta `raw/data/`
3. **Dados de saída:** arquivos limpos salvos na pasta `silver/data/`
4. **Tempo estimado:** ~2-5 minutos dependendo do tamanho do dataset
5. **Próximo passo:** Criar Star Schema (Gold Layer) para Power BI

---

**Autores:** Julio Dourado, Gustavo Rodrigues, Leonardo Lago

## 🗺️ Roteiro do Pipeline ETL

**Neste notebook, seguiremos os seguintes passos:**

1. **Importar bibliotecas** → Carregamos as ferramentas necessárias
2. **Definir funções** → Criamos funções auxiliares que usaremos depois
3. **Carregar dados** → Lemos o CSV bruto da camada Bronze
4. **Extrair ASIN** → Identificamos o código único de cada produto
5. **Deduplicar** → Removemos produtos repetidos (mantemos o mais recente)
6. **Converter tipos** → Transformamos textos em números, datas, etc.
7. **Criar features de BI** → Extraímos marca, categoria, calculamos descontos, faixas de preço, quality score, receita, flags
8. **Preencher vazios** → Usamos estratégias inteligentes para dados faltantes
9. **Selecionar colunas** → Mantemos só o que é relevante para BI (sem features de ML)
10. **Validar** → Checamos a qualidade do resultado
11. **Salvar** → Exportamos os dados limpos para a camada Silver

**Vamos começar! 🚀**

---

## 1. Configuração & Importações

**Começamos importando as bibliotecas necessárias** para trabalhar com dados tabulares, textos e arquivos.

In [66]:
# Bibliotecas para manipulação de dados
import pandas as pd  # Trabalhar com tabelas
import numpy as np   # Operações matemáticas
import re            # Trabalhar com textos e padrões
import os            # Manipular arquivos e pastas
import unicodedata   # Normalizar caracteres especiais
from datetime import datetime

# Configurações de exibição do pandas
pd.set_option('display.max_columns', None)  # Mostrar todas as colunas
pd.set_option('display.max_colwidth', 80)   # Largura máxima das colunas

In [67]:
# Caminhos dos arquivos
INPUT_FILE = '../raw/data/dados_brutos.csv'  # Dados brutos (entrada)
OUTPUT_DIR = '../silver/data'  # Pasta onde vão os dados limpos (saída)
OUTPUT_FILE_CSV = 'amazon_products_cleaned_enhanced.csv'  # Arquivo CSV de saída

## 2. Definindo Funções Auxiliares

**Agora definimos as funções que usaremos ao longo do pipeline.** Cada função tem uma responsabilidade específica de limpeza ou extração de dados.

### 2.1 Extração do ASIN

**Criamos uma função para extrair o ASIN (código único de 10 caracteres) de cada produto.**  
Precisamos desse código pois ele será essencial para identificar e remover duplicatas mais à frente.

In [68]:
def extract_asin(url):
    """
    Extract 10-char ASIN from Amazon product URLs.
    
    Examples:
        '/dp/B08N5WRWNW/' → 'B08N5WRWNW'
        '/gp/product/B09G9HD6PD' → 'B09G9HD6PD'
    """
    if pd.isna(url) or url == '':
        return None
    
    url_str = str(url).strip()
    
    # Pattern 1: /dp/ASIN or /gp/product/ASIN
    match = re.search(r'/(?:dp|gp/product|product)/([A-Z0-9]{10})(?:[/?]|$)', url_str, re.IGNORECASE)
    if match:
        return match.group(1).upper()
    
    # Pattern 2: Any 10-char alphanumeric (conservative fallback)
    if 'amazon.' in url_str.lower():
        match = re.search(r'(?:^|[^A-Z0-9])([A-Z0-9]{10})(?:$|[^A-Z0-9])', url_str, re.IGNORECASE)
        if match:
            candidate = match.group(1).upper()
            if re.fullmatch(r'[A-Z0-9]{10}', candidate):
                return candidate
    
    return None

### 2.2 Extração da Marca

**Definimos a lógica para identificar a marca de cada produto analisando seu título.**  
Precisamos filtrar palavras genéricas (stopwords) para capturar apenas o nome real da marca.

In [69]:
# Palavras que devem ser ignoradas (não são marcas)
STOPWORDS = {'the', 'a', 'an', 'new', 'latest', '2025', '2024', 'portable', 
             'wireless', 'with', 'for', 'and', 'by', 'from', 'brand', 'official'}

def extract_brand(title):
    """
    Extrai a marca do título do produto.
    
    Exemplos:
        'Samsung Galaxy S21 - 128GB' → 'samsung'
        'boAt Rockerz 450 | Bluetooth' → 'boat'
        'Apple iPhone 13 Pro Max' → 'apple'
    """
    if pd.isna(title) or title == '':
        return 'unknown'
    
    # Normaliza o texto (remove acentos, caracteres especiais)
    title_clean = unicodedata.normalize('NFKC', str(title)).strip()
    
    # Pega a primeira parte do título (antes de -, |, :, etc)
    segment = re.split(r'[-–|:()\\[,/]', title_clean, maxsplit=1)[0]
    
    # Caso especial: "Marca: XYZ"
    brand_match = re.search(r'^\\s*(?:brand|manufacturer)\\s*[:\\-]\\s*([A-Za-z0-9\\-\\+\\. ]{2,})', 
                            segment, flags=re.IGNORECASE)
    if brand_match:
        segment = brand_match.group(1)
    
    # Extrai palavras alfanuméricas
    tokens = re.findall(r'[A-Za-z0-9\\+\\.\\-]+', segment)
    
    if not tokens:
        return 'unknown'
    
    # Pega a primeira palavra que não seja stopword
    candidate = tokens[0].lower()
    if candidate in STOPWORDS or len(candidate) <= 1:
        candidate = tokens[1].lower() if len(tokens) > 1 else 'unknown'
    
    return candidate

### 2.3 Inferência de Categoria

**Construímos uma função que classifica produtos em categorias usando palavras-chave.**  
Isso nos permitirá segmentar análises por tipo de produto (Laptop, Audio, Mobile, etc.).

In [70]:
def infer_category(title):
    """
    Infer product category from title keywords.
    """
    if pd.isna(title) or str(title).strip() == '':
        return 'Other'

    title_norm = unicodedata.normalize('NFKC', str(title)).lower()
    title_norm = re.sub(r"[^a-z0-9\s]", " ", title_norm)
    title_norm = re.sub(r"\s+", " ", title_norm).strip()

    if title_norm == "":
        return 'Other'

    category_rules = [
        # Core electronics
        (r'\b(laptops?|notebooks?|macbooks?|chromebooks?|ultrabooks?|gaming\s*laptops?|surface\s*laptops?)\b', 'Laptop'),
        (r'\b(headphones?|headsets?|earbuds?|earphones?|tws|ear\s*buds?|neckbands?|speakers?|soundbars?|subwoofers?|woofers?|microphones?|mics?|lavalier|airpods?|earpods?|buds)\b', 'Audio'),
        (r'\b(cameras?|dslr|mirrorless|webcams?|action\s*cams?|gopro|camcorders?|instax|polaroid|security\s*cameras?|dash\s*cams?|picture\s*frames?)\b', 'Camera'),
        (r'\b(phones?|iphone\s?\d*|android|smartphones?|oneplus|samsung\s*galaxy|mobiles?|pixel|motorola|nokia|xiaomi|redmi|oppo|vivo|realme|infinix|tecno)\b', 'Mobile'),
        (r'\b(tablets?|ipads?|tab\b|galaxy\s*tabs?|surface\s*pros?|kindles?|fire\s*tablets?|fire\s*hd|ereaders?)\b', 'Tablet'),
        (r'\b(ssd|solid\s*state\s*drives?|hard\s*drives?|hdd|micro\s*sd|sd\s*cards?|flash\s*drives?|pen\s*drives?|pendrives?|memory\s*cards?|thumb\s*drives?|external\s*drives?|portable\s*drives?|nas|nvme|ram|memory\s*modules?)\b', 'Storage'),
        (r'\b(smartwatches?|smart\s*watches?|fitness\s*bands?|smart\s*bands?|fitbands?|wearables?|fitbit|garmin|amazfit)\b', 'Wearable'),
        (r'\b(router|routers|modems?|mesh|wifi\s*(?:system|router|kit)|range\s*extenders?|extenders?|repeater|boosters?|ethernet\s*(?:switch|adapter|hub)|access\s*point|powerline)\b', 'Networking'),
        (r'\b(processors?|cpus?|ryzen|intel\s*(?:core|pentium|celeron)|motherboards?|mainboards?|coolers?|heatsinks?|power\s*supplies?|psus?|graphics\s*cards?|gpus?|gpu)\b', 'Components'),
        (r'\b(mice|mouse|keyboards?|monitors?|adapters?|cables?|chargers?|hubs?|docks?|power\s*banks?|usb|stylus|cases?|covers?|stands?|mounts?|tripods?|gimbals?|screen\s*protectors?|cleaning\s*kits?|cooling\s*pads?|webcams?)\b', 'Accessory'),
        (r'\b(tvs?|televisions?|projectors?|smart\s*tvs?|oled|qled|uhd|4k\s*tvs?|8k\s*tvs?)\b', 'TV/Display'),
        (r'\b(playstation|xbox|ps\d|controllers?|gaming|consoles?|nintendo|switch|steam\s*deck|joysticks?|vr\s*headsets?|oculus|meta\s*quest)\b', 'Gaming'),
        # Peripherals & printing
        (r'\b(printers?|scanners?|inkjet|laserjet|label\s*makers?|plotters?)\b', 'Printing'),
        (r'\b(ink|toners?|cartridges?|drums?|refills?)\b', 'Printing Supplies'),
        # Power & smart home
        (r'\b(batteries?|power\s*banks?|alkaline|chargers?|charging\s*stations?|power\s*stations?|surge\s*protectors?|ups)\b', 'Power'),
        (r'\b(smart\s*plugs?|smart\s*bulbs?|smart\s*lights?|smart\s*locks?|doorbells?|alexa|echo|ring\s*cameras?|smart\s*displays?|homekit|smart\s*thermostats?|smart\s*home)\b', 'Smart Home'),
        # Appliances & office
        (r'\b(vacuums?|air\s*fryers?|blenders?|coffee\s*makers?|microwaves?|refrigerators?|fridges?|dishwashers?|washers?|dryers?|humidifiers?|purifiers?)\b', 'Home Appliance'),
        (r'\b(drones?|quad\s*copters?|fpv\s*drones?|mini\s*drones?)\b', 'Drone'),
        (r'\b(calculators?|ti\s*\d+|graphing\s*calculator)\b', 'Calculator/Office'),
        (r'\b(laminators?|laminating|tapes?|pencils?|pens?|markers?|notebooks?|stationery|paper|envelopes?|folders?|binders?|highlighters?|staplers?|labels?)\b', 'Office Supplies'),
        (r'\b(chairs?|desks?|standing\s*desks?|office\s*chairs?)\b', 'Office Furniture'),
        (r'\b(pet\s*pads?|litter|dog\s*pads?)\b', 'Pet Supplies'),
    ]

    for pattern, category in category_rules:
        if re.search(pattern, title_norm):
            return category

    return 'Other'

### 2.4 Parsing do Target (Unidades Vendidas)

**Desenvolvemos uma função robusta para converter textos de vendas em números.**  
Precisamos lidar com diversos formatos: "6K+", "menos de 100", "novo no mercado", etc.

In [71]:
def parse_units_sold(text):
    """
    Parse 'bought_in_last_month' column robustly.
    
    Cases:
        '6K+ bought in past month' → 6000.0
        '1.5k bought' → 1500.0
        '300+ bought' → 300.0
        'Less than 100 bought' → 50.0 (midpoint heuristic)
        'New to market' / 'Just launched' → 0.0
        Invalid → NaN
    """
    if pd.isna(text) or text == '':
        return np.nan
    
    text_clean = unicodedata.normalize('NFKC', str(text)).strip().lower()
    
    # Case 1: New products (zero sales)
    zero_phrases = ['new to market', 'just launched', 'be the first', 
                    'no sales yet', 'recently added']
    if any(phrase in text_clean for phrase in zero_phrases):
        return 0.0
    
    # Case 2: "Less than X"
    match_less = re.search(r'less\\s+than\\s+(\\d+(?:\\.\\d+)?)\\s*([km]?)', text_clean)
    if match_less:
        base_number = float(match_less.group(1))
        suffix = match_less.group(2)
        
        if suffix == 'k':
            base_number *= 1000
        elif suffix == 'm':
            base_number *= 1_000_000
        
        return base_number * 0.5  # Midpoint heuristic
    
    # Case 3: "6K+" or "1.5k" or "300+"
    match_num = re.search(r'(\\d+(?:\\.\\d+)?)\\s*([km]?)\\s*\\+?', text_clean)
    if match_num:
        base_number = float(match_num.group(1))
        suffix = match_num.group(2)
        
        if suffix == 'k':
            return base_number * 1000
        elif suffix == 'm':
            return base_number * 1_000_000
        else:
            return base_number
    
    return np.nan

### 2.5 Conversão de Preço & Extração de Cupom

**Criamos funções para limpar valores monetários e extrair descontos de cupons.**  
Precisamos transformar strings como "$1,299.99" em números puros para análises posteriores.

In [72]:
def convert_to_float(value):
    """Convert price strings to float."""
    if pd.isna(value) or value == '':
        return np.nan
    
    value = str(value).strip()
    match = re.search(r'(\\d+(?:,\\d{3})*(?:\\.\\d+)?)', value)
    
    if not match:
        return np.nan
    
    number_str = match.group(1).replace(',', '')
    return float(number_str)


def extract_coupon_percentage(coupon_text):
    """Extract coupon discount percentage."""
    if pd.isna(coupon_text) or coupon_text == '' or 'No Coupon' in str(coupon_text):
        return 0.0
    
    match = re.search(r'(\\d+(?:\\.\\d+)?)%', str(coupon_text))
    if match:
        return float(match.group(1))
    return 0.0

## 3. Carregando os Dados Brutos

**Iniciamos o processo ETL carregando o arquivo CSV com dados não processados.**  
Verificamos o tamanho do dataset para entender o volume de trabalho pela frente.

In [73]:
print("="*80)
print("🚀 ETL BRONZE → SILVER (ENHANCED)")
print("="*80)

print("\n📥 Loading Bronze data...")
df = pd.read_csv(INPUT_FILE)
print(f"   ✅ Loaded: {df.shape[0]:,} rows × {df.shape[1]} columns")
print(f"   💾 Memory: {df.memory_usage(deep=True).sum() / 1024**2:.2f} MB")

rows_initial = len(df)
df.head()

🚀 ETL BRONZE → SILVER (ENHANCED)

📥 Loading Bronze data...


   ✅ Loaded: 42,675 rows × 16 columns
   💾 Memory: 67.24 MB


,title,rating,number_of_reviews,bought_in_last_month,current/discounted_price,price_on_variant,listed_price,is_best_seller,is_sponsored,is_couponed,buy_box_availability,delivery_details,sustainability_badges,image_url,product_url,collected_at
0,"BOYA BOYALINK 2 Wireless Lavalier Microphone for iPhone Camera Android, Mini...",4.6 out of 5 stars,375,300+ bought in past month,89.68,basic variant price: 2.4GHz,$159.00,No Badge,Sponsored,Save 15% with coupon,Add to cart,"Delivery Mon, Sep 1",Carbon impact,https://m.media-amazon.com/images/I/71pAqiVEs3L._AC_UL320_.jpg,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxNDQ2OjE3NTU4MDAwNjg6c3BfYXRmX2Jy...,2025-08-21 11:14:29
1,"LISEN USB C to Lightning Cable, 240W 4 in 1 Charging Cable 6.6FT, Chubby USB...",4.3 out of 5 stars,"2,457",6K+ bought in past month,9.99,basic variant price: nan,$15.99,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Fri, Aug 29",NaN,https://m.media-amazon.com/images/I/61nbF6aVIPL._AC_UL320_.jpg,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxNDQ2OjE3NTU4MDAwNjg6c3BfYXRmX2Jy...,2025-08-21 11:14:29
2,"DJI Mic 2 (2 TX + 1 RX + Charging Case), Wireless Lavalier Microphone, Intel...",4.6 out of 5 stars,"3,044",2K+ bought in past month,314.00,basic variant price: nan,$349.00,No Badge,Sponsored,No Coupon,Add to cart,"Delivery Mon, Sep 1",NaN,https://m.media-amazon.com/images/I/61h78MEXojL._AC_UL320_.jpg,/sspa/click?ie=UTF8&spc=MTo4NzEzNDY2NTQ5NDYxNDQ2OjE3NTU4MDAwNjg6c3BfYXRmX2Jy...,2025-08-21 11:14:29
3,"Apple AirPods Pro 2 Wireless Earbuds, Active Noise Cancellation, Hearing Aid...",4.6 out of 5 stars,"35,882",10K+ bought in past month,NaN,basic variant price: $162.24,No Discount,Best Seller,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61SUj2aKoEL._AC_UL320_.jpg,/Apple-Cancellation-Transparency-Personalized-High-Fidelity/dp/B0D1XD1ZV3/re...,2025-08-21 11:14:29
4,"Apple AirTag 4 Pack. Keep Track of and find Your Keys, Wallet, Luggage, Back...",4.8 out of 5 stars,"28,988",10K+ bought in past month,NaN,basic variant price: $72.74,No Discount,No Badge,Organic,No Coupon,NaN,NaN,NaN,https://m.media-amazon.com/images/I/61bMNCeAUAL._AC_UL320_.jpg,/Apple-MX542LL-A-AirTag-Pack/dp/B0D54JZTHY/ref=sr_1_5?dib=eyJ2IjoiMSJ9.avmZl...,2025-08-21 11:14:29


## 4. Extraindo ASIN e Removendo Duplicatas

**Aqui começamos a limpar os dados extraindo o código ASIN de cada produto.**  
Precisamos fazer isso ANTES de qualquer filtragem pois produtos duplicados podem ter informações diferentes em cada coleta.

In [74]:
print("\\n📌 Extraindo ASIN (código único do produto)...")

# Aplicamos a função extract_asin em todas as URLs
df['asin'] = df['product_url'].apply(extract_asin)

# Contamos quantos produtos conseguimos identificar
with_asin = df['asin'].notna().sum()
without_asin = df['asin'].isna().sum()

print(f"   ✅ Com ASIN: {with_asin:,} ({with_asin/len(df)*100:.1f}%)")
print(f"   ⚠️  Sem ASIN: {without_asin:,} ({without_asin/len(df)*100:.1f}%)")

if with_asin > 0:
    unique_asins = df['asin'].nunique()
    duplicates = with_asin - unique_asins
    print(f"   📊 ASINs únicos: {unique_asins:,}")
    print(f"   🔄 Duplicatas detectadas: {duplicates:,}")

\n📌 Extraindo ASIN (código único do produto)...
   ✅ Com ASIN: 35,114 (82.3%)
   ⚠️  Sem ASIN: 7,561 (17.7%)
   📊 ASINs únicos: 8,377
   🔄 Duplicatas detectadas: 26,737


In [75]:
print("\\n🔄 Removendo duplicatas por ASIN...")

# Separamos produtos com ASIN (podemos deduplicar) dos sem ASIN (mantemos todos)
df_with_asin = df[df['asin'].notna()].copy()
df_without_asin = df[df['asin'].isna()].copy()

if len(df_with_asin) > 0:
    # Convertemos a data de coleta para poder ordenar
    df_with_asin['collected_at'] = pd.to_datetime(df_with_asin['collected_at'], errors='coerce')
    
    # Ordenamos: produtos iguais (mesmo ASIN) ficam juntos, com o mais recente primeiro
    df_with_asin = df_with_asin.sort_values(['asin', 'collected_at'], ascending=[True, False])
    
    # Removemos duplicatas: para cada ASIN, mantemos apenas a primeira linha (mais recente)
    df_dedup = df_with_asin.drop_duplicates(subset=['asin'], keep='first')
    
    # Juntamos de volta: produtos deduplicados + produtos sem ASIN
    df = pd.concat([df_dedup, df_without_asin], axis=0, ignore_index=True)
    
    # Calculamos quantas linhas foram removidas
    rows_removed = rows_initial - len(df)
    print(f"   ✅ Antes: {rows_initial:,}")
    print(f"   ✅ Depois: {len(df):,}")
    print(f"   🗑️  Duplicatas removidas: {rows_removed:,} ({rows_removed/rows_initial*100:.1f}%)")

\n🔄 Removendo duplicatas por ASIN...
   ✅ Antes: 42,675
   ✅ Depois: 15,938
   🗑️  Duplicatas removidas: 26,737 (62.7%)


## 5. Convertendo Tipos de Dados

**Agora convertemos cada coluna para seu tipo adequado.**  
Precisamos fazer isso para que operações matemáticas funcionem corretamente e para criar features temporais úteis.

In [76]:
print("\\n🔢 Convertendo tipos de dados...")

# Convertemos strings de data/hora em objetos datetime do pandas
df['collected_at'] = pd.to_datetime(df['collected_at'], errors='coerce')

# Criamos features temporais úteis para análises
df['date'] = df['collected_at'].dt.normalize()  # Só a data (00:00:00)
df['time'] = df['collected_at'].dt.time  # Só o horário
df['hour'] = df['collected_at'].dt.hour  # Hora do dia (0-23)
df['day_of_week'] = df['collected_at'].dt.dayofweek  # 0=Segunda, 6=Domingo
df['day_name'] = df['collected_at'].dt.day_name()  # Nome completo do dia

print(f"   ✅ Período dos dados: {df['date'].min()} até {df['date'].max()}")

\n🔢 Convertendo tipos de dados...
   ✅ Período dos dados: 2025-08-21 00:00:00 até 2025-08-30 00:00:00


In [77]:
# Limpamos e convertemos avaliações e reviews
# Extrai número da string, converte para float com tratamento de erros, e limita entre 0 e 5
df['rating'] = pd.to_numeric(df['rating'].str.extract(r'([\d\.]+)', expand=False), errors='coerce').clip(0, 5)

# Remove vírgulas dos reviews e converte para inteiro
df['number_of_reviews'] = df['number_of_reviews'].str.replace(',', '', regex=False).fillna('0').astype(int)

print("   ✅ Avaliações e reviews convertidos")

   ✅ Avaliações e reviews convertidos


In [78]:
# Convertemos a variável alvo (nossa variável de previsão)
print("   🎯 Convertendo unidades vendidas (target)...")

# Aplicamos a função de parsing em todos os textos de vendas
df['units_sold'] = df['bought_in_last_month'].apply(parse_units_sold)

parsed_count = df['units_sold'].notna().sum()
print(f"      ✅ {parsed_count:,} registros com target válido ({parsed_count/len(df)*100:.1f}%)")

   🎯 Convertendo unidades vendidas (target)...
      ✅ 0 registros com target válido (0.0%)


In [79]:
# Convertemos todas as colunas de preço para números
df['current/discounted_price'] = df['current/discounted_price'].str.replace('$', '', regex=False).str.replace(',', '', regex=False).apply(lambda x: float(x) if x and x != '0' else np.nan)
df['listed_price'] = df['listed_price'].apply(convert_to_float)
df['price_on_variant'] = df['price_on_variant'].apply(convert_to_float)

print("   ✅ Preços convertidos para números")

   ✅ Preços convertidos para números


In [80]:
# Convertemos flags de texto para booleanos (True/False)
df['is_best_seller'] = df['is_best_seller'].apply(lambda x: True if str(x).strip() == 'Best Seller' else False).astype(bool)
df['is_sponsored'] = df['is_sponsored'].apply(lambda x: True if str(x).strip() == 'Sponsored' else False).astype(bool)
df['buy_box_availability'] = df['buy_box_availability'].apply(lambda x: True if str(x).strip() == 'Add to cart' else False).astype(bool)

# Garantimos que colunas de texto são tipo string
df['title'] = df['title'].astype('string')
df['time'] = df['time'].astype('string')

print("   ✅ Conversão de tipos concluída")

   ✅ Conversão de tipos concluída


## 6. Criando Novas Features (Engenharia de Atributos)

**Passamos agora para a criação de colunas derivadas que agregarão valor às análises.**  
Vamos extrair informações escondidas nos dados brutos e calcular métricas importantes.

### 6.1 Extração de Marca

**Aplicamos a função de extração de marca em todos os produtos.**  
Isso nos dá visibilidade de quantas marcas diferentes existem no dataset.

In [81]:
print("\n🏷️  Feature engineering...")
print("   📦 Extracting brands...")
df['brand'] = df['title'].apply(extract_brand)
print(f"      ✅ {df['brand'].nunique():,} unique brands")

# Show top brands
df['brand'].value_counts().head(10)


🏷️  Feature engineering...
   📦 Extracting brands...
      ✅ 628 unique brands


brand
duracell     888
energizer    812
asurion      665
hp           629
kodak        556
belkin       468
trx          456
rca          373
amscope      323
acer         288
Name: count, dtype: int64

### 6.2 Classificação por Categoria

**Classificamos cada produto em categorias usando a função de inferência.**  
Verificamos a distribuição para entender quais tipos de produtos são mais comuns.

In [82]:
print("   📂 Inferring categories...")
df['category'] = df['title'].apply(infer_category)
category_dist = df['category'].value_counts(dropna=False)
print(f"      ✅ {len(category_dist)} categories identified")
for cat, count in category_dist.head(10).items():
    print(f"         - {cat}: {count:,} ({count/len(df):.1%})")

other_pct = (category_dist.get('Other', 0) / len(df)) * 100
print(f"      ℹ️  'Other' share: {other_pct:.1f}%")

# Visualize category distribution with absolute counts and share
category_summary = (
    category_dist.to_frame(name='count')
    .assign(share=lambda s: (s['count'] / len(df) * 100).round(2))
)
category_summary.head(15)

   📂 Inferring categories...


      ✅ 22 categories identified
         - Audio: 2,529 (15.9%)
         - Other: 2,294 (14.4%)
         - Camera: 2,174 (13.6%)
         - Power: 1,930 (12.1%)
         - Accessory: 1,593 (10.0%)
         - Laptop: 1,251 (7.8%)
         - Mobile: 1,198 (7.5%)
         - Printing: 609 (3.8%)
         - Storage: 526 (3.3%)
         - Networking: 335 (2.1%)
      ℹ️  'Other' share: 14.4%


,count,share
category,,
Audio,2529,15.87
Other,2294,14.39
Camera,2174,13.64
Power,1930,12.11
Accessory,1593,9.99
Laptop,1251,7.85
Mobile,1198,7.52
Printing,609,3.82
Storage,526,3.30


### 6.3 Unificação de Preços

**Criamos uma coluna única de preço final usando lógica em cascata.**  
Tentamos: 1º preço com desconto → 2º preço da variante → 3º preço original. Assim maximizamos a cobertura.

In [83]:
print("   💰 Building final_price (waterfall logic)...")
df['final_price'] = df['current/discounted_price'].fillna(df['price_on_variant']).fillna(df['listed_price'])

df['price_source'] = np.select(
    [
        df['current/discounted_price'].notna(),
        df['current/discounted_price'].isna() & df['price_on_variant'].notna(),
        df['current/discounted_price'].isna() & df['price_on_variant'].isna() & df['listed_price'].notna()
    ],
    ['discounted', 'variant', 'original'],
    default='none'
)

missing_price = df['final_price'].isna().sum()
print(f"      ⚠️  Missing prices: {missing_price:,} ({missing_price/len(df)*100:.1f}%)")

   💰 Building final_price (waterfall logic)...
      ⚠️  Missing prices: 3,031 (19.0%)


### 6.4 Preenchimento Inteligente de Preços Faltantes

**Usamos estatísticas inteligentes para preencher preços que ainda estão vazios.**  
Priorizamos: 1º mediana da mesma marca+categoria → 2º mediana da marca → 3º mediana da categoria → 4º mediana geral.

In [84]:
if missing_price > 0:
    print("   🔧 Applying smart price imputation...")
    
    median_brand_cat = df.groupby(['brand', 'category'])['final_price'].median()
    median_brand = df.groupby('brand')['final_price'].median()
    median_cat = df.groupby('category')['final_price'].median()
    global_median = df['final_price'].median()
    
    def impute_price(row):
        if pd.notna(row['final_price']):
            return row['final_price'], 'original'
        
        key = (row['brand'], row['category'])
        if key in median_brand_cat and pd.notna(median_brand_cat[key]):
            return median_brand_cat[key], 'imputed_brand_cat'
        
        if row['brand'] in median_brand and pd.notna(median_brand[row['brand']]):
            return median_brand[row['brand']], 'imputed_brand'
        
        if row['category'] in median_cat and pd.notna(median_cat[row['category']]):
            return median_cat[row['category']], 'imputed_category'
        
        return global_median, 'imputed_global'
    
    imputed_data = df.apply(impute_price, axis=1, result_type='expand')
    df['final_price'] = imputed_data[0]
    df['price_imputation_tier'] = imputed_data[1]
    
    imputed_count = (df['price_imputation_tier'] != 'original').sum()
    print(f"      ✅ {imputed_count:,} prices imputed")
    
    # Show imputation breakdown
    df['price_imputation_tier'].value_counts()

   🔧 Applying smart price imputation...


      ✅ 3,031 prices imputed


### 6.5 Cálculo de Descontos

**Calculamos a porcentagem de desconto comparando preço original vs. preço final.**  
Criamos também faixas de desconto para facilitar análises segmentadas (0-10%, 10-20%, etc.).

In [85]:
print("   🎁 Discount engineering...")
df['discount_pct'] = np.where(
    (df['listed_price'].notna()) & (df['listed_price'] > 0) & (df['final_price'].notna()),
    (df['listed_price'] - df['final_price']) / df['listed_price'] * 100,
    0.0
).clip(0, 95)

df['discount_bucket'] = pd.cut(
    df['discount_pct'],
    bins=[-0.1, 0, 10, 20, 30, 50, 95],
    labels=['No Discount', '0-10%', '10-20%', '20-30%', '30-50%', '50%+']
)

df['has_discount'] = df['discount_pct'] > 0

print(f"      ✅ {df['has_discount'].sum():,} products with discounts")
df['discount_bucket'].value_counts().sort_index()

   🎁 Discount engineering...
      ✅ 0 products with discounts


discount_bucket
No Discount    15938
0-10%              0
10-20%             0
20-30%             0
30-50%             0
50%+               0
Name: count, dtype: int64

### 6.6 Identificação de Cupons

**Extraímos informações sobre cupons de desconto disponíveis.**  
Marcamos produtos que têm cupom e calculamos a porcentagem de desconto adicional.

In [86]:
print("   🎟️  Processing coupons...")
df['coupon_discount_pct'] = df['is_couponed'].apply(extract_coupon_percentage)
df['has_coupon'] = (df['coupon_discount_pct'] > 0).astype(bool)
coupon_count = df['has_coupon'].sum()
print(f"      ✅ {coupon_count:,} products with coupons")

df[df['has_coupon']][['title', 'coupon_discount_pct']].head()

   🎟️  Processing coupons...
      ✅ 0 products with coupons


,title,coupon_discount_pct


### 6.7 Faixas de Preço (Price Tiers)

**Categorizamos produtos por faixa de preço para análise de mercado.**  
Isso facilita identificar quais segmentos (Budget, Premium, Luxury) têm melhor desempenho.

In [87]:
print("   💎 Criando faixas de preço...")

def create_price_tier(price):
    """Categoriza produtos por faixa de preço para segmentação de mercado"""
    if pd.isna(price):
        return 'Unknown'
    elif price < 20:
        return 'Budget (< $20)'
    elif price < 50:
        return 'Economy ($20-50)'
    elif price < 100:
        return 'Mid-Range ($50-100)'
    elif price < 200:
        return 'Premium ($100-200)'
    elif price < 500:
        return 'High-End ($200-500)'
    else:
        return 'Luxury ($500+)'

df['price_tier'] = df['final_price'].apply(create_price_tier)

# Mostra distribuição
print(f"      ✅ Distribuição por faixa de preço:")
tier_dist = df['price_tier'].value_counts()
for tier, count in tier_dist.items():
    print(f"         - {tier}: {count:,} ({count/len(df)*100:.1f}%)")

def create_price_tier(price):
    """Categoriza produtos por faixa de preço para segmentação de mercado"""
    if pd.isna(price):
        return 'Unknown'
    elif price < 20:
        return 'Budget (< $20)'
    elif price < 50:
        return 'Economy ($20-50)'
    elif price < 100:
        return 'Mid-Range ($50-100)'
    elif price < 200:
        return 'Premium ($100-200)'
    elif price < 500:
        return 'High-End ($200-500)'
    else:
        return 'Luxury ($500+)'

df['price_tier'] = df['final_price'].apply(create_price_tier)

# Mostra distribuição
print(f"      ✅ Distribuição por faixa de preço:")
tier_dist = df['price_tier'].value_counts()
for tier, count in tier_dist.items():
    print(f"         - {tier}: {count:,} ({count/len(df)*100:.1f}%)")

   💎 Criando faixas de preço...
      ✅ Distribuição por faixa de preço:
         - Economy ($20-50): 3,963 (24.9%)
         - Premium ($100-200): 3,454 (21.7%)
         - Budget (< $20): 2,972 (18.6%)
         - Mid-Range ($50-100): 2,698 (16.9%)
         - High-End ($200-500): 1,707 (10.7%)
         - Luxury ($500+): 1,144 (7.2%)
      ✅ Distribuição por faixa de preço:
         - Economy ($20-50): 3,963 (24.9%)
         - Premium ($100-200): 3,454 (21.7%)
         - Budget (< $20): 2,972 (18.6%)
         - Mid-Range ($50-100): 2,698 (16.9%)
         - High-End ($200-500): 1,707 (10.7%)
         - Luxury ($500+): 1,144 (7.2%)


### 6.8 Score de Qualidade do Produto

**Combinamos rating e reviews em uma métrica única de 0-100.**  
Útil para criar quadrantes de análise (alta qualidade + baixa venda = oportunidade!).

In [88]:
print("   ⭐ Calculando quality score...")

def calculate_quality_score(row):
    """
    Combina rating + social proof para score 0-100
    50 pontos: rating normalizado (0-5 → 0-50)
    50 pontos: reviews em escala log (0-10K+ → 0-50)
    """
    if pd.isna(row['rating']) or pd.isna(row['number_of_reviews']):
        return None
    
    # Normalizar rating (0-5 → 0-50 pontos)
    rating_score = (row['rating'] / 5.0) * 50
    
    # Normalizar reviews (log scale, 0-50 pontos)
    # 0 reviews = 0 pts, 10K+ reviews = 50 pts
    review_score = min(50, (np.log1p(row['number_of_reviews']) / np.log1p(10000)) * 50)
    
    return round(rating_score + review_score, 2)

df['quality_score'] = df.apply(calculate_quality_score, axis=1)

# Estatísticas
valid_scores = df['quality_score'].notna().sum()
print(f"      ✅ {valid_scores:,} produtos com quality score")
print(f"      📊 Score médio: {df['quality_score'].mean():.2f}")
print(f"      📊 Score mediano: {df['quality_score'].median():.2f}")

   ⭐ Calculando quality score...


      ✅ 15,557 produtos com quality score
      📊 Score médio: 77.45
      📊 Score mediano: 78.50


### 6.9 Receita Estimada

**Calculamos a receita potencial multiplicando unidades vendidas por preço.**  
Métrica essencial para identificar produtos que geram mais valor financeiro.

In [89]:
print("   💰 Calculando receita estimada...")

# Receita = Unidades Vendidas × Preço Final
df['estimated_revenue'] = df['units_sold'] * df['final_price']

# Estatísticas
valid_revenue = df['estimated_revenue'].notna().sum()
total_revenue = df['estimated_revenue'].sum()

print(f"      ✅ {valid_revenue:,} produtos com receita calculada")
print(f"      💵 Receita total estimada: ${total_revenue:,.2f}")
print(f"      💵 Receita média por produto: ${df['estimated_revenue'].mean():,.2f}")

   💰 Calculando receita estimada...
      ✅ 0 produtos com receita calculada
      💵 Receita total estimada: $0.00
      💵 Receita média por produto: $nan


### 6.10 Flag: Produto Promovível

**Identificamos produtos prontos para promoção baseado em critérios de qualidade.**  
Rating ≥ 4.0, Reviews ≥ 100, Vendas ≥ 200, Disponível para compra = Produto promovível!

In [90]:
print("   🎯 Identificando produtos promovíveis...")

def is_promotable(row):
    """
    Produto promovível se atende critérios de qualidade:
    - Rating >= 4.0 (boa avaliação)
    - Reviews >= 100 (social proof)
    - Units sold >= 200 (demanda comprovada)
    - Buy box disponível
    """
    if pd.isna(row['rating']) or pd.isna(row['number_of_reviews']) or pd.isna(row['units_sold']):
        return False
    
    return (
        row['rating'] >= 4.0 and
        row['number_of_reviews'] >= 100 and
        row['units_sold'] >= 200 and
        row['buy_box_availability'] == True
    )

df['is_promotable'] = df.apply(is_promotable, axis=1)

promotable_count = df['is_promotable'].sum()
print(f"      ✅ {promotable_count:,} produtos promovíveis ({promotable_count/len(df)*100:.1f}%)")

# Top 5 marcas com mais produtos promovíveis
top_promotable = df[df['is_promotable']].groupby('brand').size().sort_values(ascending=False).head(5)
print(f"      🏆 Top 5 marcas promovíveis:")
for brand, count in top_promotable.items():
    print(f"         - {brand}: {count:,} produtos")

   🎯 Identificando produtos promovíveis...


      ✅ 0 produtos promovíveis (0.0%)
      🏆 Top 5 marcas promovíveis:


## 7. Selecionando e Renomeando Colunas Finais

**Organizamos o dataset final selecionando apenas colunas relevantes.**  
Renomeamos algumas para nomes mais intuitivos e descartamos colunas temporárias ou redundantes.

In [91]:
print("\n📋 Selecionando e renomeando colunas finais...")

# Renomeamos colunas para nomes mais claros no Power BI
df = df.rename(columns={
    'number_of_reviews': 'review_count',
    'current/discounted_price': 'discounted_price',
    'listed_price': 'original_price',
    'bought_in_last_month': 'bought_in_last_month_raw',
    'is_couponed': 'is_couponed_raw',
    'is_best_seller': 'best_seller_badge',
    'is_sponsored': 'sponsored_badge',
    'buy_box_availability': 'available_for_purchase',
    'has_discount': 'is_discounted',
    'has_coupon': 'has_active_coupon',
    'units_sold': 'units_sold_last_month',
    'estimated_revenue': 'revenue_last_month'
})

# Colunas para manter (orientadas a BI, sem features de ML)
columns_to_keep = [
    # IDs
    'asin',
    
    # Produto
    'title', 'brand', 'category',
    
    # Métricas de Qualidade
    'rating', 'review_count', 'quality_score',
    
    # Preços
    'final_price', 'original_price', 'price_tier',
    'price_source',
    
    # Descontos
    'discount_pct', 'discount_bucket', 'is_discounted',
    
    # Cupons
    'has_active_coupon', 'coupon_discount_pct',
    
    # Badges/Flags
    'best_seller_badge', 'sponsored_badge', 'available_for_purchase',
    'is_promotable',
    
    # Métricas de Vendas (MEASURES para Power BI)
    'units_sold_last_month', 'revenue_last_month',
    
    # Tempo
    'date', 'hour', 'day_of_week', 'day_name', 'collected_at',
    
    # Auditoria
    'bought_in_last_month_raw', 'is_couponed_raw'
]

# Adiciona price_imputation_tier se existir
if 'price_imputation_tier' in df.columns:
    columns_to_keep.append('price_imputation_tier')

# Filtra apenas colunas que existem
columns_to_keep = [col for col in columns_to_keep if col in df.columns]
df_silver = df[columns_to_keep].copy()

print(f"   ✅ {len(columns_to_keep)} colunas selecionadas")
print(f"   📊 Shape final: {df_silver.shape}")
print(f"\n   🎯 Features para BI:")
print(f"      - Faixas de preço: price_tier")
print(f"      - Score de qualidade: quality_score")
print(f"      - Receita estimada: revenue_last_month")
print(f"      - Produto promovível: is_promotable")
print(f"\n   ❌ Features de ML removidas:")
print(f"      - log1p_reviews, log1p_price, log1p_units_sold")

df_silver.head()


📋 Selecionando e renomeando colunas finais...
   ✅ 30 colunas selecionadas
   📊 Shape final: (15938, 30)

   🎯 Features para BI:
      - Faixas de preço: price_tier
      - Score de qualidade: quality_score
      - Receita estimada: revenue_last_month
      - Produto promovível: is_promotable

   ❌ Features de ML removidas:
      - log1p_reviews, log1p_price, log1p_units_sold


,asin,title,brand,category,rating,review_count,quality_score,final_price,original_price,price_tier,price_source,discount_pct,discount_bucket,is_discounted,has_active_coupon,coupon_discount_pct,best_seller_badge,sponsored_badge,available_for_purchase,is_promotable,units_sold_last_month,revenue_last_month,date,hour,day_of_week,day_name,collected_at,bought_in_last_month_raw,is_couponed_raw,price_imputation_tier
0,1426215649,Destinations of a Lifetime: 225 of the World's Most Amazing Places,destinations,Other,4.7,5602,93.85,20.93,NaN,Economy ($20-50),discounted,0.0,No Discount,False,False,0.0,False,False,False,False,NaN,NaN,2025-08-21,11,3,Thursday,2025-08-21 11:43:48,List:,No Coupon,original
1,9792351833,"Brother Genuine P-Touch TZe White Print on Black Label Tape (TZe335), Lamina...",brother,Printing,5.0,5,59.73,17.99,NaN,Budget (< $20),discounted,0.0,No Discount,False,False,0.0,False,False,True,False,NaN,NaN,2025-08-21,12,3,Thursday,2025-08-21 12:01:33,600+ bought in past month,No Coupon,original
2,B000001OKK,Maxell 108527 Optimally Designed Flat Packs with Low Noise Surface 90 Min Re...,maxell,Other,4.6,4293,91.41,7.51,NaN,Budget (< $20),discounted,0.0,No Discount,False,False,0.0,False,False,True,False,NaN,NaN,2025-08-21,12,3,Thursday,2025-08-21 12:12:41,500+ bought in past month,No Coupon,original
3,B000001OM5,Maxell – Pro 190048 CD-340 Laser Lens Cleaner - Safe & Effective CD Player &...,maxell,Other,4.2,14432,92.00,9.50,NaN,Budget (< $20),none,0.0,No Discount,False,False,0.0,False,False,False,False,NaN,NaN,2025-08-21,11,3,Thursday,2025-08-21 11:40:46,3K+ bought in past month,No Coupon,imputed_brand_cat
4,B00000DMFD,"Hasbro Gaming Mouse Trap Kids Board Game, Family Board Games for Kids, Kids ...",hasbro,Accessory,4.5,9206,94.55,24.99,NaN,Economy ($20-50),discounted,0.0,No Discount,False,False,0.0,False,False,True,False,NaN,NaN,2025-08-21,11,3,Thursday,2025-08-21 11:50:06,800+ bought in past month,No Coupon,original


## 8. Validando a Qualidade dos Dados

**Realizamos verificações finais para garantir que o processo ETL foi bem-sucedido.**  
Checamos completude, cobertura do target, distribuições e outras métricas de qualidade.

In [92]:
print("\n🔍 VALIDAÇÃO FINAL")
print("="*70)

# Completude geral
total_cells = df_silver.shape[0] * df_silver.shape[1]
missing_cells = df_silver.isna().sum().sum()
completeness = ((total_cells - missing_cells) / total_cells * 100)
print(f"📊 Completude dos dados: {completeness:.2f}%")

# Cobertura de métricas críticas
target_coverage = df_silver['units_sold_last_month'].notna().sum()
print(f"\n🎯 Cobertura de métricas:")
print(f"   - Unidades vendidas: {target_coverage:,} ({target_coverage/len(df_silver)*100:.1f}%)")

price_coverage = df_silver['final_price'].notna().sum()
print(f"   - Preços: {price_coverage:,} ({price_coverage/len(df_silver)*100:.1f}%)")

revenue_coverage = df_silver['revenue_last_month'].notna().sum()
print(f"   - Receita estimada: {revenue_coverage:,} ({revenue_coverage/len(df_silver)*100:.1f}%)")

quality_coverage = df_silver['quality_score'].notna().sum()
print(f"   - Quality score: {quality_coverage:,} ({quality_coverage/len(df_silver)*100:.1f}%)")

# Dimensões de negócio
print(f"\n📦 Dimensões de negócio:")
print(f"   - ASINs únicos: {df_silver['asin'].nunique():,}")
print(f"   - Marcas: {df_silver['brand'].nunique():,}")
print(f"   - Categorias: {df_silver['category'].nunique():,}")
print(f"   - Faixas de preço: {df_silver['price_tier'].nunique():,}")
print(f"   - Dias de coleta: {df_silver['date'].nunique():,}")

# Flags de qualidade
print(f"\n🚩 Flags de qualidade/ação:")
print(f"   - Best Seller: {df_silver['best_seller_badge'].sum():,} ({df_silver['best_seller_badge'].sum()/len(df_silver)*100:.1f}%)")
print(f"   - Patrocinados: {df_silver['sponsored_badge'].sum():,} ({df_silver['sponsored_badge'].sum()/len(df_silver)*100:.1f}%)")
print(f"   - Com cupom ativo: {df_silver['has_active_coupon'].sum():,} ({df_silver['has_active_coupon'].sum()/len(df_silver)*100:.1f}%)")
print(f"   - Com desconto: {df_silver['is_discounted'].sum():,} ({df_silver['is_discounted'].sum()/len(df_silver)*100:.1f}%)")
print(f"   - PROMOVÍVEIS: {df_silver['is_promotable'].sum():,} ({df_silver['is_promotable'].sum()/len(df_silver)*100:.1f}%)")

# Métricas de negócio
print(f"\n💰 Métricas de negócio:")
total_units = df_silver['units_sold_last_month'].sum()
total_revenue = df_silver['revenue_last_month'].sum()
avg_price = df_silver['final_price'].mean()
avg_quality = df_silver['quality_score'].mean()

print(f"   - Total unidades vendidas: {total_units:,.0f}")
print(f"   - Receita total estimada: ${total_revenue:,.2f}")
print(f"   - Preço médio: ${avg_price:,.2f}")
print(f"   - Quality score médio: {avg_quality:.2f}/100")

print("="*70)


🔍 VALIDAÇÃO FINAL
📊 Completude dos dados: 87.79%

🎯 Cobertura de métricas:
   - Unidades vendidas: 0 (0.0%)
   - Preços: 15,938 (100.0%)
   - Receita estimada: 0 (0.0%)
   - Quality score: 15,557 (97.6%)

📦 Dimensões de negócio:
   - ASINs únicos: 8,377
   - Marcas: 628
   - Categorias: 22
   - Faixas de preço: 6
   - Dias de coleta: 6

🚩 Flags de qualidade/ação:
   - Best Seller: 256 (1.6%)
   - Patrocinados: 7,011 (44.0%)
   - Com cupom ativo: 0 (0.0%)
   - Com desconto: 0 (0.0%)
   - PROMOVÍVEIS: 0 (0.0%)

💰 Métricas de negócio:
   - Total unidades vendidas: 0
   - Receita total estimada: $0.00
   - Preço médio: $160.09
   - Quality score médio: 77.45/100


In [93]:
# Data quality overview
df_silver.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 15938 entries, 0 to 15937
Data columns (total 30 columns):
 #   Column                    Non-Null Count  Dtype         
---  ------                    --------------  -----         
 0   asin                      8377 non-null   object        
 1   title                     15938 non-null  string        
 2   brand                     15938 non-null  object        
 3   category                  15938 non-null  object        
 4   rating                    15557 non-null  float64       
 5   review_count              15938 non-null  int64         
 6   quality_score             15557 non-null  float64       
 7   final_price               15938 non-null  float64       
 8   original_price            0 non-null      float64       
 9   price_tier                15938 non-null  object        
 10  price_source              15938 non-null  object        
 11  discount_pct              15938 non-null  float64       
 12  discount_bucket   

In [94]:
# Statistical summary
df_silver.describe()

,rating,review_count,quality_score,final_price,original_price,discount_pct,coupon_discount_pct,units_sold_last_month,revenue_last_month,date,hour,day_of_week,collected_at
count,15557.00000,15938.000000,15557.000000,15938.000000,0.0,15938.0,15938.0,0.0,0.0,15938,15938.000000,15938.000000,15938
mean,4.43581,4891.758376,77.446985,160.089992,NaN,0.0,0.0,NaN,NaN,2025-08-23 14:59:26.118710016,12.397666,3.543230,2025-08-24 03:55:46.023403008
min,1.00000,0.000000,13.760000,2.490000,NaN,0.0,0.0,NaN,NaN,2025-08-21 00:00:00,0.000000,0.000000,2025-08-21 11:14:29
25%,4.30000,68.000000,67.460000,25.990000,NaN,0.0,0.0,NaN,NaN,2025-08-21 00:00:00,11.000000,3.000000,2025-08-21 11:44:13
50%,4.50000,492.500000,78.500000,69.950000,NaN,0.0,0.0,NaN,NaN,2025-08-21 00:00:00,11.000000,3.000000,2025-08-21 12:13:05
75%,4.70000,3520.000000,88.690000,159.950000,NaN,0.0,0.0,NaN,NaN,2025-08-25 00:00:00,12.000000,4.000000,2025-08-25 11:24:55
max,5.00000,865598.000000,99.000000,4699.000000,NaN,0.0,0.0,NaN,NaN,2025-08-30 00:00:00,22.000000,6.000000,2025-08-30 19:56:33
std,0.36197,19898.314715,14.149593,291.977748,NaN,0.0,0.0,NaN,NaN,NaN,4.869223,1.474357,NaN


### Exemplos: Top Produtos Promovíveis

In [95]:
# Top 10 produtos promovíveis por receita
print("🏆 Top 10 Produtos Promovíveis (por receita):\n")

promotable_products = df_silver[df_silver['is_promotable'] == True].copy()

if len(promotable_products) > 0:
    top_promotable = promotable_products.nlargest(10, 'revenue_last_month')[
        ['title', 'brand', 'category', 'price_tier', 'rating', 
         'review_count', 'quality_score', 'units_sold_last_month', 'revenue_last_month']
    ]
    
    display(top_promotable)
    
    print(f"\n💡 Insight: Esses {len(top_promotable)} produtos têm:")
    print(f"   - Alta qualidade (rating ≥ 4.0)")
    print(f"   - Social proof (≥100 reviews)")
    print(f"   - Demanda comprovada (≥200 vendas)")
    print(f"   - Disponíveis para compra")
    print(f"\n   → Candidatos ideais para promoção e destaque!")
else:
    print("Nenhum produto atende aos critérios de promovibilidade.")

🏆 Top 10 Produtos Promovíveis (por receita):

Nenhum produto atende aos critérios de promovibilidade.


## 9. Salvando os Dados Processados

**Finalmente, salvamos o dataset limpo na camada Silver em formato CSV:**  
- **CSV:** formato universal, fácil de abrir em Excel, Power BI ou outras ferramentas de análise

In [96]:
print("\n💾 Saving to Silver layer...")

os.makedirs(OUTPUT_DIR, exist_ok=True)

# CSV
csv_path = os.path.join(OUTPUT_DIR, OUTPUT_FILE_CSV)
df_silver.to_csv(csv_path, index=False, encoding='utf-8')
print(f"   ✅ CSV saved: {csv_path}")
print(f"      Size: {os.path.getsize(csv_path) / 1024**2:.2f} MB")


💾 Saving to Silver layer...


   ✅ CSV saved: ../silver/data/amazon_products_cleaned_enhanced.csv
      Size: 5.45 MB


## 10. Resumo do Processo ETL

**Visualizamos o resultado final do pipeline completo.**  
Comparamos entrada vs. saída, verificamos perdas de dados e confirmamos que está tudo pronto para as próximas etapas: análises exploratórias, modelagem preditiva ou criação de dashboards.

In [97]:
print("\n" + "="*80)
print("🎉 ETL COMPLETO - SILVER LAYER PRONTO PARA GOLD")
print("="*80)

print(f"\n📥 ENTRADA (Bronze):")
print(f"   - Linhas: {rows_initial:,}")
print(f"   - Formato: CSV bruto, dados não estruturados")

print(f"\n📤 SAÍDA (Silver):")
print(f"   - Linhas: {len(df_silver):,}")
print(f"   - Colunas: {len(df_silver.columns)}")
print(f"   - Formato: CSV otimizado")

print(f"\n🔧 TRANSFORMAÇÕES APLICADAS:")
dedup_count = rows_initial - len(df_silver)
print(f"   ✅ Deduplicação por ASIN: {dedup_count:,} registros removidos")
print(f"   ✅ Extração de marca e categoria")
print(f"   ✅ Imputação inteligente de preços")
print(f"   ✅ Parsing de unidades vendidas")
print(f"   ✅ Engenharia de descontos e cupons")

print(f"\n🆕 FEATURES CRIADAS PARA BI:")
print(f"   💎 price_tier - Faixas de preço para segmentação")
print(f"   ⭐ quality_score - Métrica composta (rating + reviews)")
print(f"   💰 revenue_last_month - Receita estimada por produto")
print(f"   🎯 is_promotable - Flag de produtos prontos para promoção")

print(f"\n✅ PRÓXIMOS PASSOS:")
print(f"   1. Criar Star Schema (Gold Layer)")
print(f"   2. Popular dimensões: dim_produto, dim_tempo, dim_preco")
print(f"   3. Popular fato: fato_vendas")
print(f"   4. Conectar ao Power BI")
print(f"   5. Criar dashboards de análise")

print(f"\n🎯 OBJETIVO DE NEGÓCIO:")
print(f"   Identificar produtos com melhor desempenho para:")
print(f"   - Decisões de estoque")
print(f"   - Estratégias de promoção")
print(f"   - Destaque na plataforma")

print("\n" + "="*80)
print("✨ Dados prontos para análise de performance de produtos! ✨")
print("="*80)


🎉 ETL COMPLETO - SILVER LAYER PRONTO PARA GOLD

📥 ENTRADA (Bronze):
   - Linhas: 42,675
   - Formato: CSV bruto, dados não estruturados

📤 SAÍDA (Silver):
   - Linhas: 15,938
   - Colunas: 30
   - Formato: CSV otimizado

🔧 TRANSFORMAÇÕES APLICADAS:
   ✅ Deduplicação por ASIN: 26,737 registros removidos
   ✅ Extração de marca e categoria
   ✅ Imputação inteligente de preços
   ✅ Parsing de unidades vendidas
   ✅ Engenharia de descontos e cupons

🆕 FEATURES CRIADAS PARA BI:
   💎 price_tier - Faixas de preço para segmentação
   ⭐ quality_score - Métrica composta (rating + reviews)
   💰 revenue_last_month - Receita estimada por produto
   🎯 is_promotable - Flag de produtos prontos para promoção

✅ PRÓXIMOS PASSOS:
   1. Criar Star Schema (Gold Layer)
   2. Popular dimensões: dim_produto, dim_tempo, dim_preco
   3. Popular fato: fato_vendas
   4. Conectar ao Power BI
   5. Criar dashboards de análise

🎯 OBJETIVO DE NEGÓCIO:
   Identificar produtos com melhor desempenho para:
   - Decisões 

---

## 📝 Changelog - Versão Otimizada para BI

### ✅ Features ADICIONADAS (voltadas para Power BI):

1. **`price_tier`** - Categoriza produtos em 6 faixas de preço
   - Budget (< $20), Economy ($20-50), Mid-Range ($50-100), Premium ($100-200), High-End ($200-500), Luxury ($500+)
   - Uso: Segmentação de mercado, análise por faixa de preço

2. **`quality_score`** - Score composto 0-100
   - 50 pontos: rating normalizado
   - 50 pontos: social proof (reviews em escala log)
   - Uso: Quadrantes de análise, identificar "hidden gems"

3. **`revenue_last_month`** - Receita estimada
   - Cálculo: units_sold × final_price
   - Uso: KPI principal, ranking de produtos por valor gerado

4. **`is_promotable`** - Flag booleana
   - Critérios: rating ≥ 4.0, reviews ≥ 100, vendas ≥ 200, disponível para compra
   - Uso: Filtro para identificar produtos prontos para promoção

### ❌ Features REMOVIDAS (voltadas para ML):

1. **`log1p_reviews`** - Transformação logarítmica de reviews
2. **`log1p_price`** - Transformação logarítmica de preço  
3. **`log1p_units_sold`** - Transformação logarítmica de vendas

**Motivo:** Estas transformações são úteis para estabilizar variância em modelos de ML, mas não agregam valor em dashboards de BI. Power BI trabalha melhor com valores naturais.

### 🔄 Colunas RENOMEADAS (para clareza no Power BI):

- `number_of_reviews` → `review_count`
- `is_best_seller` → `best_seller_badge`
- `is_sponsored` → `sponsored_badge`
- `buy_box_availability` → `available_for_purchase`
- `has_discount` → `is_discounted`
- `has_coupon` → `has_active_coupon`
- `units_sold` → `units_sold_last_month`
- `estimated_revenue` → `revenue_last_month`

### 📊 Resultado Final:

**Colunas no Silver:** ~32 colunas  
**Orientação:** Business Intelligence / Power BI  
**Próximo passo:** Criar Star Schema (Gold Layer)

---